# 03 ? Storage, Exact Search, ANN, and Persistence

This notebook connects vector search concepts from `03_vector_search_and_indexing` to the responsibilities of a vector database. The earlier notebooks explained similarity metrics, brute-force search, IVF, HNSW, and product quantization. Here we ask: what data does a database keep, what work does an index save, what quality/performance tradeoffs does approximate search introduce, and what must survive a restart?

This notebook stays database-agnostic. Configuration names and persistence details differ across products; the concepts are transferable.

## Learning goals

By the end, you should be able to:

- separate the stored records from the index used to find candidate records;
- explain exact search as a baseline and ANN as a speed/recall tradeoff;
- describe the high-level search process of HNSW and IVF;
- estimate raw vector storage and identify what that estimate leaves out;
- distinguish durable records, rebuildable indexes, and backups;
- explain how inserts, updates, deletes, and restarts affect database state.

## 1. Connect the previous topics to database storage

A vector database may need to manage several kinds of state:

1. **Record identity and payload** ? IDs, text or references, metadata, and application fields.
2. **Vector values** ? the embeddings used for similarity comparisons.
3. **Search structures** ? indexes that help locate likely neighbors without scanning every vector.
4. **Durability state** ? logs, snapshots, or other mechanisms to recover accepted writes after a crash or restart.
5. **Operational metadata** ? collection configuration, index state, versions, and other product-specific information.

These may live in separate files, memory structures, or services. A database may keep the original full-precision vector even if its search index uses a compressed representation. Product details vary; keep the logical roles separate.

## 2. Exact search: the reference point

With exact nearest-neighbor search, the query is compared with every vector in the eligible search set, then the best K results are returned. For N vectors of dimension D, a dense scan is roughly proportional to **N ? D** work per query (plus selection and filtering overhead).

Exact search is useful because it gives a ground-truth ranking for a fixed metric, query vector, and filter. It can be practical for small collections or highly optimized hardware. As N and query volume grow, scanning every vector can become too costly for the latency or compute budget.

The earlier brute-force notebook covers the math and implementation in more depth; here exact search is the baseline against which an ANN index can be evaluated.

In [1]:
# A tiny exact-search reference implementation using Euclidean distance.
# This is educational Python, not an optimized database implementation.
from math import sqrt

def euclidean(a, b):
    return sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

def exact_top_k(query, items, k):
    scored = [(item["id"], euclidean(query, item["vector"])) for item in items]
    return sorted(scored, key=lambda pair: pair[1])[:k]

toy_items = [
    {"id": "p0", "vector": [0.0, 0.0]},
    {"id": "p1", "vector": [1.0, 0.0]},
    {"id": "p2", "vector": [0.0, 1.0]},
    {"id": "p3", "vector": [4.0, 4.0]},
]
exact_top_k([0.8, 0.2], toy_items, k=3)

[('p1', 0.282842712474619),
 ('p0', 0.8246211251235323),
 ('p2', 1.1313708498984762)]

## 3. ANN: search likely candidates

Approximate nearest-neighbor (ANN) search uses an index to avoid comparing the query with every vector. It usually trades some retrieval quality or additional index cost for lower query work. Depending on the index and settings, an ANN result may omit one of the true exact top-K neighbors.

A useful evaluation loop is:

1. Run exact search on a representative sample to establish the ground truth.
2. Run ANN with chosen index and search settings on the same queries and filters.
3. Compare overlap/recall at K and measure latency and resource use.
4. Adjust index/build/search settings and repeat.

Recall is not the only quality measure, but it is a direct way to check whether approximate search found the exact neighbors. A database may report a distance/score; its direction and meaning depend on the metric and API.

### Recall@K

For a query, define:

**Recall@K = (number of ANN results that also appear in the exact top-K) / K**

If exact top-5 is `{a,b,c,d,e}` and ANN returns `{a,b,c,x,y}`, recall@5 is 3/5 = 0.6. Across a query set, average the per-query recall. Compare latency on the same workload; a recall number without the matching query distribution and filter conditions is incomplete.

In [2]:
def recall_at_k(exact_ids, approximate_ids, k):
    exact_set = set(exact_ids[:k])
    approximate_set = set(approximate_ids[:k])
    return len(exact_set & approximate_set) / k

recall_at_k(["a", "b", "c", "d", "e"], ["a", "b", "c", "x", "y"], k=5)

0.6

## 4. HNSW: navigate a proximity graph

HNSW (Hierarchical Navigable Small World) organizes vectors in a graph with multiple levels. Higher levels are sparser navigation layers; the bottom layer contains the full set of indexed items. Search starts in a sparse layer, moves toward closer nodes, descends, and explores candidates in the lower layer.

The graph lets a query navigate toward nearby regions without scanning all records. It also has costs: graph edges consume memory, index construction takes time, and search parameters affect recall and latency. A high-level picture:

```text
Sparse upper layer:        A -------- F
                            |          |
Middle layer:        A --- C --- F --- H
                      \   |   /       |
Bottom layer:       A-B-C-D-E-F-G-H-I-J   (dense, many connections)
```

Common HNSW parameter names include `M` (graph connectivity), `ef_construction` (candidate breadth during build), and `ef_search` (candidate breadth during query). Names and exact behavior vary. In general, more connectivity or wider exploration can improve recall but costs memory, build time, or query time. Consult the database's own documentation before tuning.

### What to measure for HNSW

- **Build time:** how long does ingestion/index construction take?
- **Index memory:** how much extra memory do graph links and index structures use?
- **Query latency:** report percentiles such as p50 and p95, not only a mean.
- **Recall@K:** compare with exact search using the same filters and query set.
- **Update behavior:** are inserts searchable immediately, buffered, or incorporated asynchronously?

A higher `ef_search`-like setting often explores more candidates. It may improve recall while increasing latency. Do not select settings from a single query.

## 5. IVF: partition the vector space

IVF (Inverted File) divides the vector space into coarse regions, often represented by centroids. During search, the query is compared with centroids; only the nearest selected partitions (lists) are scanned for candidates.

- `nlist` commonly refers to the number of partitions/lists.
- `nprobe` commonly refers to how many lists are searched for a query.

With too few probes, the true nearest vector may be in an unvisited list. Increasing probes examines more candidates and can improve recall, but reduces the amount of work saved. Building useful partitions also requires training or learning the coarse quantizer on representative vectors. Exact terminology and details vary by implementation.

If product quantization is combined with IVF, vectors or residuals may be compressed for candidate scoring. Compression changes the memory/accuracy tradeoff; product quantization was covered in your previous indexing notebooks.

### HNSW and IVF in database terms

| Question | HNSW | IVF |
|---|---|---|
| How are candidates organized? | Navigable graph | Coarse partitions/lists |
| Main query-time tradeoff | How broadly to explore the graph | How many lists to probe |
| Typical extra cost | Graph memory and construction | Training/partitioning and list storage |
| What can hurt recall? | Search explores too few candidates | Query probes too few or poor partitions |
| What changes when data shifts? | Graph may need ongoing maintenance | Coarse partitions may become unrepresentative |

This is a conceptual comparison, not a universal product benchmark. Dataset shape, filters, hardware, and implementation can change the outcome.

## 6. Storage size: vectors are only one part

A first raw-vector estimate is:

**raw vector bytes ? number of vectors ? dimensions ? bytes per value**

For example, one million 768-dimensional vectors stored as 32-bit floats require about 3.07 billion bytes (about 2.86 GiB) for vector values alone. This excludes IDs, payload/text, alignment, index structures, replicas, temporary build data, allocator overhead, and backups. Actual database memory and disk use can be substantially different.

Compression reduces some storage costs, but may affect recall or require retaining a higher-precision copy. Also distinguish **disk footprint** from **resident memory**: an index can be persisted on disk while some or all of its working structures are loaded in RAM for low-latency search.

In [3]:
def raw_vector_bytes(count, dimensions, bytes_per_value=4):
    return count * dimensions * bytes_per_value

size_bytes = raw_vector_bytes(count=1_000_000, dimensions=768)
size_gib = size_bytes / (1024 ** 3)
size_bytes, round(size_gib, 2)

(3072000000, 2.86)

This estimate is a lower-level planning aid, not a capacity plan. To plan a real collection, measure representative records and index settings on the target database and hardware. Include growth, replicas, backups, and ingestion-time headroom.

## 7. Persistence: what survives a restart?

Persistence means accepted data survives process or machine restarts according to a stated durability guarantee. Ask exactly what is durable:

- Are vectors and payloads written to durable storage before a write is acknowledged?
- Is there a write-ahead log (WAL) or another recovery mechanism?
- Is the search index persisted, rebuilt after restart, or both?
- Does the service load the index fully into RAM, partially, or on demand?
- What does a snapshot contain, and how is it restored?
- What is the recovery point and recovery time after a failure?

A **rebuildable index** is not necessarily a backup. If the canonical vectors, IDs, and payloads are lost, rebuilding the index cannot restore them. Keep the original source documents and/or durable canonical records according to your recovery design.

### WAL, snapshots, and recovery (generic concepts)

A write-ahead log records changes so a system can recover operations after a crash. A snapshot captures a point-in-time database state or a product-specific subset of it. On restart, a system may load persisted index files, replay a log, rebuild some index structures, or combine these steps.

These are broad database concepts, not a guarantee that every vector database uses the same mechanism. For a chosen product, learn its documented write acknowledgment semantics, snapshot contents, restore procedure, compatibility constraints, and backup consistency guarantees.

## 8. Inserts, updates, and deletes meet the index

A stored record and a search index are related but distinct. When records change, the system must make the index reflect the new searchable state. Implementations may update structures synchronously, buffer changes, use segments, or merge/rebuild data in the background.

Ask the database-specific questions:

- When is a newly written vector visible to search?
- Does an update replace the old vector and payload together?
- When does a delete stop appearing in results, and when is space reclaimed?
- Can a large bulk load build the index more efficiently than many small writes?
- Is there a compaction/optimization step, and what resources does it need?
- How do filters interact with ANN candidate selection and recall?

Do not infer immediate visibility or physical disk reclamation solely from a successful API response. Learn the product's consistency and lifecycle behavior.

## 9. A practical index evaluation plan

Use the same dataset and query set to compare configurations. Record:

1. Collection size, vector dimension, metric, embedding model, and hardware.
2. Whether queries are filtered and how selective those filters are.
3. Build time and index size/disk use.
4. Query latency distribution (at least p50 and p95) under representative concurrency.
5. Recall@K against exact results for a representative query sample.
6. Update throughput and how quickly new/changed records become searchable.
7. Memory use during steady-state search and during index build/restore.

Change one setting at a time when learning its effect. A faster benchmark with lower recall may or may not be acceptable; set an application-specific quality target before tuning.

## 10. Choosing a starting approach

A reasonable learning sequence is:

- Use exact search to establish correctness and a small-scale baseline.
- Try the database's recommended ANN index for the collection size and workload.
- Measure recall and latency using your own query distribution and filters.
- Increase search effort if recall is too low; evaluate its latency cost.
- Consider compression or sharding only after measuring the bottleneck and understanding the quality impact.
- Test restart, backup, and restore behavior separately from query-speed tuning.

There is no universally best index. Data distribution, filters, write rate, dimensionality, hardware, and operational requirements all matter.

## Knowledge check

1. Why is exact search useful even if production will use ANN?
2. What does an HNSW search-effort parameter generally trade against recall?
3. In IVF, what happens when a query probes more lists? What is the cost?
4. Why is `N ? D ? 4` not the complete memory estimate for 32-bit vectors?
5. If an index can be rebuilt, why might you still need backups?
6. What should you measure when a database acknowledges an insert but the new record is not immediately returned by ANN search?

### Suggested answers

1. It provides a ground-truth result set for checking ANN recall and a correctness baseline.
2. Broader search usually improves the chance of finding true neighbors but requires more work and can raise latency.
3. More lists mean more candidates are searched, which can improve recall while reducing the work saved by partitioning.
4. It excludes payloads, IDs, index structures, replicas, runtime overhead, and other state; disk and resident memory also differ.
5. Rebuilding an index requires the underlying vectors and records. An index backup/rebuild plan does not recover lost canonical data or source documents by itself.
6. Check documented write visibility/consistency, index update behavior, filters, and whether background indexing is pending; measure the delay as part of the workload.

## Summary

Exact search gives a reference ranking; ANN indexes such as HNSW and IVF reduce query work by navigating or partitioning the search space, introducing tunable recall and resource tradeoffs. A database must manage the records and the index as data changes, and persistence has to protect the canonical vectors and payloads as well as any index state required for recovery.

**Next:** `04_qdrant_hands_on.ipynb` will apply the record model and persistence concepts in Qdrant.